<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 그래프 학습 · 추천 시스템 · 08. PinSage: random-walk 기반 추천 이웃

## Graph Convolutional Neural Networks for Web-Scale Recommender Systems

- **원 논문:** [Graph Convolutional Neural Networks for Web-Scale Recommender Systems](https://arxiv.org/abs/1806.01973)
- **저자 / 발표:** Rex Ying, Ruining He, Kaifeng Chen, Pong Eksombatchai, William L. Hamilton, Jure Leskovec · KDD (2018)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** GraphSAGE, bipartite graph, ranking loss
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** user-item graph에서 item graph를 만들고 random-walk visit importance로 weighted convolution을 수행합니다.

**축소하거나 생략한 부분:** Pinterest 30억 node/180억 edge와 MapReduce 대신 6 user×9 item을 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.2, importance-based neighborhoods | random-walk visit count top-T | L1 weight·top-k |
| §3.2, Algorithm 1 line 1 | importance pooling | weighted mean |
| Algorithm 1 lines 2–3 | concat transform와 L2 normalize | unit norm |
| §3.3, Eq. (1) | max-margin ranking | positive/negative score gap |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** interaction [U,I] → item graph [I,I] → user/item embedding [U/I,D]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$h_v^{k}=\sigma\!\left(W^k[h_v^{k-1}\Vert\sum_{u\in\mathcal N(v)}\alpha_u h_u^{k-1}]\right)$$

- **기호 정의:** v는 item, α는 random-walk visit 기반 중요도, N(v)는 top-T 이웃입니다.
- **수식의 역할:** bipartite walk로 얻은 중요 이웃만 가중 집계해 웹 규모 추천을 근사합니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** Task P의 `preprocess_graph`, 논문 전용 클래스의
  `forward`, `compute_loss`, `training_step`이 §3.2, importance-based neighborhoods을 구조화합니다.
- **입출력 shape:** user-item [U, I] → item affinity [I, I] → user/item score [U, I]
- **평가:** BPR 또는 margin loss 감소와 held-out Recall@3
- **원문 대비 한계:** Pinterest 30억 node/180억 edge와 MapReduce 대신 6 user×9 item을 사용합니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 논문 주장 확인 실험

- **원문 위치:** §3.2, importance-based neighborhoods
- **이 셀의 책임:** random-walk visit count top-T
- **Tensor shape 계약:** interaction [U,I] → item graph [I,I] → user/item embedding [U/I,D]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** L1 weight·top-k. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
user_item = torch.tensor(raw["user_item"], dtype=torch.float32)
item_graph = user_item.T @ user_item
item_graph.fill_diagonal_(0)
item_graph = (item_graph > 0).float()
assert user_item.ndim == 2 and item_graph.shape == (
    user_item.shape[1],
    user_item.shape[1],
)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.2, importance-based neighborhoods
- **이 셀의 책임:** random-walk visit count top-T
- **Tensor shape 계약:** interaction [U,I] → item graph [I,I] → user/item embedding [U/I,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** L1 weight·top-k. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 08-1: item에서 시작한 짧은 walk의 visit count로 top-k 중요 이웃을 구하세요.
def importance_neighbors(adj: Tensor, start: int, walks=24, length=4, topk=3, seed=0):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.2, Algorithm 1 line 1 / Algorithm 1 lines 2–3
- **이 셀의 책임:** importance pooling / concat transform와 L2 normalize
- **Tensor shape 계약:** interaction [U,I] → item graph [I,I] → user/item embedding [U/I,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** weighted mean / unit norm. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 08-2: Algorithm 1의 importance pooling convolution을 구현하세요.
class PinSageConv(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, dim):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x, adj):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3.3, Eq. (1)
- **이 셀의 책임:** max-margin ranking
- **Tensor shape 계약:** interaction [U,I] → item graph [I,I] → user/item embedding [U/I,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** positive/negative score gap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 08-3: Eq. (1) margin ranking으로 positive co-view를 negative보다 가깝게 학습하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §3.2, importance-based neighborhoods / §3.2, Algorithm 1 line 1 / Algorithm 1 lines 2–3 / §3.3, Eq. (1)
- **이 셀의 책임:** random-walk visit count top-T / importance pooling / concat transform와 L2 normalize / max-margin ranking
- **Tensor shape 계약:** interaction [U,I] → item graph [I,I] → user/item embedding [U/I,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** L1 weight·top-k / weighted mean / unit norm / positive/negative score gap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO P-1: 논문의 핵심 연산을 아래 구조에 직접 구현하세요.
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    embedding_dim: int = 10
    walk_count: int = 24
    walk_length: int = 4
    top_neighbors: int = 3
    margin: float = 0.5
    recall_k: int = 3
    learning_rate: float = 0.03
    steps: int = 24


def preprocess_graph(
    interactions: Tensor, config: PortfolioConfig
) -> tuple[Tensor, Tensor, Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class PinsagePortfolioModel(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, item_count: int, config: PortfolioConfig):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, importance: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def compute_loss(
        self, importance: Tensor, query: Tensor, positive: Tensor, negative: Tensor
    ) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def training_step(
        self, importance: Tensor, query: Tensor, positive: Tensor, negative: Tensor
    ) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def sample_margin_triplets(
    item_graph: Tensor, step: int
) -> tuple[Tensor, Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** §3.3, Eq. (1)
- **이 셀의 책임:** max-margin ranking
- **Tensor shape 계약:** interaction [U,I] → item graph [I,I] → user/item embedding [U/I,D]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** positive/negative score gap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: PinsagePortfolioModel, importance: Tensor, item_graph: Tensor
) -> list[float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** §3.3, Eq. (1)
- **이 셀의 책임:** max-margin ranking
- **Tensor shape 계약:** interaction [U,I] → item graph [I,I] → user/item embedding [U/I,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** positive/negative score gap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: PinsagePortfolioModel,
    importance: Tensor,
    train_interactions: Tensor,
    heldout: Tensor,
) -> dict[str, float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.